# Bancada de testes — Redes Neurais no Adult Census Income

**Dataset:** [Adult Census Income](https://www.kaggle.com/datasets/uciml/adult-census-income) (UCI, censo dos EUA de 1994) · **Algoritmo:** rede neural do tipo perceptron multicamadas (MLP) · **Tarefa:** classificação binária — prever se a renda anual de uma pessoa é maior que 50 mil dólares (`>50K`).

O objetivo desta bancada é **testar, de forma controlada, as configurações da rede, as técnicas de tratamento dos dados e a engenharia de features** e descobrir qual combinação maximiza as métricas de avaliação: acurácia, precisão, recall, F1, ROC-AUC, PR-AUC e MCC.

Todo o código reutilizável (pré-processamento, rede, validação cruzada e registro dos resultados) está em [`bancada.py`](bancada.py). Este notebook apenas define e roda os experimentos.

## O que aprendemos com os notebooks de referência

Os dois notebooks analisados usam o mesmo dataset:

| Notebook | Pré-processamento | Modelo | Resultado |
|---|---|---|---|
| *EDA + Logistic Regression + PCA* | `?` → moda, `LabelEncoder`, `StandardScaler` | Regressão logística | acurácia 0,8218 (todas as features) |
| *T2-3 Adult Census Income tutorial* | `?` → moda / categoria `"null"`, `LabelEncoder`, `MinMaxScaler` | Árvore de decisão / Random Forest | acurácia 0,812 / 0,857 |

Pontos que esta bancada corrige ou investiga:

1. **Só a acurácia foi medida.** Como ~76% das pessoas ganham `<=50K`, um modelo que sempre responde `<=50K` já tem 76% de acurácia. Por isso medimos também precisão, recall, F1, ROC-AUC, PR-AUC e MCC.
2. **`LabelEncoder` em variáveis sem ordem** (ocupação, país, raça…) cria uma ordem falsa (ex.: `Sales` > `Prof-specialty`). Para árvores isso pouco importa, mas para modelos lineares e redes neurais costuma prejudicar. Testamos *one-hot*, *one-hot* com agrupamento de categorias raras e *target encoding*.
3. **Confusão entre componentes principais e colunas originais.** No notebook de PCA, a variância explicada pelo último *componente* foi usada para justificar a remoção da *coluna* `native.country` — mas cada componente é uma combinação de todas as colunas, então não há essa correspondência. Aqui o PCA é testado da forma correta: como uma etapa do pipeline que gera componentes.
4. **Vazamento de informação.** O tutorial calcula a moda do teste no próprio teste e ajusta o `LabelEncoder` com treino + teste juntos. Aqui todo o pré-processamento fica dentro de um `Pipeline` ajustado **somente nos dados de treino de cada fold**.
5. **Uma única divisão treino/validação** dá resultados ruidosos. Usamos **validação cruzada estratificada com 5 folds**, sempre com as mesmas divisões para todas as configurações (comparação justa).

## Bibliotecas

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             precision_recall_curve, roc_curve)
from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

import bancada as bc   # código da bancada (bancada.py, na mesma pasta)

RANDOM_STATE = bc.RANDOM_STATE   # 42: mesma semente em todo o notebook
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# Cores dos gráficos (paleta categórica validada para daltonismo)
AZUL, LARANJA, VERDE_AGUA, AMARELO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
CINZA_TEXTO, CINZA_GRADE = "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": CINZA_TEXTO, "axes.labelcolor": CINZA_TEXTO,
    "xtick.color": CINZA_TEXTO, "ytick.color": CINZA_TEXTO,
    "axes.grid": True, "grid.color": CINZA_GRADE, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "font.size": 10,
})

## Carregamento dos dados

A função `carregar_dados` baixa o dataset do Kaggle com `kagglehub`. Se não houver acesso ao Kaggle, ela usa um espelho do UCI no GitHub, que tem exatamente as mesmas 32.561 linhas. Os valores `?` (dado desconhecido) já são convertidos em `NaN`.

In [ ]:
df = bc.carregar_dados()
df.head()

In [ ]:
# Tipos das colunas e quantidade de valores não nulos
df.info()

## Análise exploratória

Pontos que guiam os experimentos: o desbalanceamento do alvo, onde estão os valores faltantes, a assimetria das colunas de capital e a redundância entre `education` e `education.num`.

In [ ]:
# Variável alvo: classes desbalanceadas (~76% x 24%)
print(df["income"].value_counts())
print()
print(df["income"].value_counts(normalize=True).round(3))

In [ ]:
# Valores faltantes (antes eram '?'): só em 3 colunas categóricas
faltantes = df.isna().sum()
print(faltantes[faltantes > 0])
print("\nLinhas com pelo menos um faltante:", df.isna().any(axis=1).sum(),
      f"({df.isna().any(axis=1).mean():.1%})")

In [ ]:
# Taxa de renda > 50K entre quem tem e quem não tem o valor faltante:
# se a taxa muda, "faltar" é uma informação útil (motiva a categoria própria)
alvo = (df["income"] == ">50K")
for col in ["workclass", "occupation", "native.country"]:
    print(f"{col:<15} faltante: {alvo[df[col].isna()].mean():.3f}   "
          f"preenchido: {alvo[df[col].notna()].mean():.3f}")

In [ ]:
# Colunas numéricas: capital.gain e capital.loss são quase sempre zero e muito assimétricas
print(df.describe().T.round(1))
print("\nProporção de zeros:")
print((df[["capital.gain", "capital.loss"]] == 0).mean().round(3))

In [ ]:
# education e education.num carregam a mesma informação (uma categoria = um número)
print(df.groupby("education")["education.num"].nunique().max(),
      "valor de education.num por categoria de education")

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 3.8))

# Taxa de renda > 50K por anos de estudo
taxa = alvo.groupby(df["education.num"]).mean()
eixos[0].bar(taxa.index, taxa.values, color=AZUL, width=0.75)
eixos[0].set(title="Renda > 50K por anos de estudo", xlabel="education.num", ylabel="proporção > 50K")

# Taxa por estado civil
taxa = alvo.groupby(df["marital.status"]).mean().sort_values()
eixos[1].barh(taxa.index, taxa.values, color=AZUL, height=0.7)
eixos[1].set(title="Renda > 50K por estado civil", xlabel="proporção > 50K")

# Distribuição do ganho de capital (escala log por causa da assimetria)
ganho = df.loc[df["capital.gain"] > 0, "capital.gain"]
eixos[2].hist(np.log1p(ganho), bins=40, color=AZUL, edgecolor="white")
eixos[2].set(title="log(1 + capital.gain), só valores > 0", xlabel="log(1 + capital.gain)", ylabel="pessoas")
plt.tight_layout()
plt.show()

## Protocolo experimental

1. **Teste final separado (hold-out):** 20% dos dados, estratificado, guardado até o fim. Ele **não** é usado em nenhuma escolha.
2. **Validação cruzada estratificada (5 folds)** nos 80% restantes. Em cada fold, o pré-processamento é ajustado só no treino do fold, e 10% do treino do fold são reservados para a **parada antecipada** (o treino para quando a perda nesse conjunto deixa de cair por `paciencia` épocas, e os pesos da melhor época são restaurados).
3. **Busca gulosa em etapas:** cada etapa testa variações de **um** aspecto (faltantes, codificação, escala, features, PCA, arquitetura, regularização, ativação, otimização, balanceamento) sobre a melhor configuração encontrada até então. Ao final, uma **busca aleatória** refina os hiperparâmetros da rede em conjunto.
4. **Métrica de escolha: ROC-AUC.** Ela mede a capacidade do modelo de *ordenar* as pessoas por chance de ter renda alta, sem depender do limiar de decisão. Acurácia, precisão, recall e F1 dependem do limiar (0,5 durante as etapas); por isso o **limiar é otimizado numa etapa própria**, no fim, usando as previsões fora-do-fold.
5. **Todas as métricas são registradas** em cada experimento (média dos 5 folds; `roc_auc_dp` é o desvio-padrão entre folds).

A rede é o `MLPClassifier` do scikit-learn, treinado época a época por um laço próprio (`bancada.treinar_rede`) para termos parada antecipada pela perda de validação. Opções testadas da rede: nº de camadas e neurônios, ativação (`relu`, `tanh`, `logistic`), regularização L2, otimizador (`adam` ou SGD com momentum de Nesterov), taxa de aprendizado, tamanho do lote, redução da taxa de aprendizado em platôs e pesos por classe.

In [ ]:
X, y = bc.separar_x_y(df)   # y = 1 se renda > 50K

# Teste final (20%), estratificado: só será usado na última seção
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Treino:", X_treino.shape, " Teste:", X_teste.shape)
print(f"Proporção de > 50K -> treino: {y_treino.mean():.3f}  teste: {y_teste.mean():.3f}")

In [ ]:
# Configuração inicial: reproduz o pré-processamento dos notebooks de referência
# (moda nos faltantes + LabelEncoder + StandardScaler) com uma MLP de 2 camadas
pd.Series(bc.CONFIG_BASE, name="valor").to_frame()

In [ ]:
bancada = bc.Bancada(X_treino, y_treino, n_folds=5, metrica_escolha="roc_auc")
inicio_total = time.time()

## Etapa 0 — Linha de base

A rede com o pré-processamento dos notebooks de referência. Todos os ganhos seguintes são medidos em relação a ela.

In [ ]:
tab = bancada.rodar_etapa("0. Linha de base", {"label + standard, MLP (64, 32)": {}})
tab

## Etapa 1 — Valores faltantes

- **moda:** troca o faltante pela categoria mais frequente (como nos notebooks de referência);
- **categoria:** cria a categoria `Desconhecido` — preserva a informação de que o dado faltava;
- **remover:** descarta do treino as linhas com faltantes (~7% delas). Na validação/teste, que não podem perder linhas, o faltante recebe a moda.

In [ ]:
tab = bancada.rodar_etapa("1. Faltantes", {
    "moda": {"faltantes": "moda"},
    "categoria 'Desconhecido'": {"faltantes": "categoria"},
    "remover linhas (treino)": {"faltantes": "remover"},
})
tab

## Etapa 2 — Codificação das variáveis categóricas

- **label:** um inteiro por categoria (ordem arbitrária) + padronização;
- **onehot:** uma coluna 0/1 por categoria (sem ordem falsa; ~100 colunas);
- **onehot_raras:** *one-hot*, mas categorias com menos de 0,5% das linhas viram uma só (`infrequent`) — reduz colunas quase vazias (vários países);
- **target:** cada categoria vira a proporção de renda > 50K dentro dela (calculada com validação cruzada interna para não vazar o alvo).

In [ ]:
tab = bancada.rodar_etapa("2. Codificação", {
    "label": {"codificacao": "label"},
    "onehot": {"codificacao": "onehot"},
    "onehot + raras agrupadas": {"codificacao": "onehot_raras"},
    "target encoding": {"codificacao": "target"},
})
tab

## Etapa 3 — Escala das variáveis numéricas

Redes neurais são sensíveis à escala das entradas. Além dos escalonadores, testamos aplicar **log(1 + x)** em `capital.gain` e `capital.loss` antes da padronização, para reduzir a assimetria vista na análise exploratória. `quantile` e `power` (Yeo-Johnson) também deixam as distribuições mais simétricas.

In [ ]:
features_atuais = tuple(bancada.melhor_cfg["features"])
tab = bancada.rodar_etapa("3. Escala", {
    "standard": {"escala": "standard"},
    "minmax": {"escala": "minmax"},
    "robust": {"escala": "robust"},
    "quantile (saída normal)": {"escala": "quantile"},
    "power (Yeo-Johnson)": {"escala": "power"},
    "standard + log capital": {"escala": "standard", "features": features_atuais + ("log_capital",)},
    "minmax + log capital": {"escala": "minmax", "features": features_atuais + ("log_capital",)},
})
tab

## Etapa 4 — Engenharia de features

Primeiro, cada transformação é testada **sozinha** sobre a configuração atual (sem adotar). Depois, testamos a **combinação** das que ajudaram individualmente e uma seleção gulosa (adicionando, uma a uma, as que mais ajudaram enquanto houver ganho).

| Opção | O que faz |
|---|---|
| `remover_education` | remove `education` (mesma informação de `education.num`) |
| `remover_fnlwgt` | remove o peso amostral do censo (não descreve a pessoa) |
| `capital_liquido` | cria `capital.net = capital.gain − capital.loss` |
| `flags_capital` | cria indicadores 0/1 de ter ganho e de ter perda de capital |
| `log_capital` | log(1 + x) nas colunas de capital |
| `agrupar_pais` | `native.country` → Estados Unidos × Outro |
| `agrupar_estado_civil` | 7 estados civis → Casado, Separado, Viúvo, Solteiro |
| `agrupar_setor` | `workclass` → Governo, Privado, Autônomo, Autônomo-inc, Sem renda |
| `faixas_idade_horas` | faixas de idade e de horas semanais (categóricas) |
| `interacoes` | idade × escolaridade e horas × escolaridade |

In [ ]:
OPCOES_FEATURES = ["remover_education", "remover_fnlwgt", "capital_liquido", "flags_capital",
                   "log_capital", "agrupar_pais", "agrupar_estado_civil", "agrupar_setor",
                   "faixas_idade_horas", "interacoes"]

atuais = tuple(bancada.melhor_cfg["features"])
candidatas = [o for o in OPCOES_FEATURES if o not in atuais]
variacoes = {"atual": {}}
variacoes.update({f"+ {o}": {"features": atuais + (o,)} for o in candidatas})

tab_individual = bancada.rodar_etapa("4a. Features (uma a uma)", variacoes, adotar=False)
tab_individual

In [ ]:
# Ganho de ROC-AUC de cada opção em relação à configuração atual
ref = tab_individual.loc["atual", "roc_auc"]
ganhos = (tab_individual["roc_auc"].drop("atual") - ref).sort_values(ascending=False)
ganhos.index = ganhos.index.str.replace("+ ", "", regex=False)
print(ganhos.round(4).to_string())
uteis = [o for o in ganhos.index if ganhos[o] > 0]
print("\nOpções que ajudaram sozinhas:", uteis)

In [ ]:
# Seleção gulosa: adiciona as opções na ordem do ganho individual,
# mantendo cada uma só se a ROC-AUC da combinação melhorar
selecionadas, melhor_auc = list(atuais), ref
for opcao in uteis:
    cfg = {**bancada.melhor_cfg, "features": tuple(selecionadas + [opcao])}
    linha = bancada.avaliar(cfg, "4b. Features (seleção gulosa)", f"+ {opcao}")
    manter = linha["roc_auc"] > melhor_auc
    print(f"  + {opcao:<22} ROC-AUC={linha['roc_auc']:.4f}  -> {'mantém' if manter else 'descarta'}")
    if manter:
        selecionadas.append(opcao)
        melhor_auc = linha["roc_auc"]
print("\nSeleção gulosa:", selecionadas)

In [ ]:
tab = bancada.rodar_etapa("4c. Features (combinações)", {
    "atual": {},
    "todas as úteis juntas": {"features": tuple(dict.fromkeys(list(atuais) + uteis))},
    "seleção gulosa": {"features": tuple(selecionadas)},
    "todas as opções": {"features": tuple(OPCOES_FEATURES)},
})
tab

## Etapa 5 — PCA

Testamos a ideia do notebook de referência da forma correta: o PCA entra no pipeline **depois** da codificação e da escala, e a rede recebe apenas os componentes que explicam 99%, 95% ou 90% da variância.

In [ ]:
tab = bancada.rodar_etapa("5. PCA", {
    "sem PCA": {"pca": None},
    "PCA 99% da variância": {"pca": 0.99},
    "PCA 95% da variância": {"pca": 0.95},
    "PCA 90% da variância": {"pca": 0.90},
})
tab

## Etapa 6 — Arquitetura (profundidade e largura)

Número de camadas ocultas e de neurônios por camada, do mais simples (uma camada com 16 neurônios) ao mais complexo (3 camadas, 256-128-64).

In [ ]:
ARQUITETURAS = [(16,), (32,), (64,), (128,), (256,),
                (32, 16), (64, 32), (128, 64), (256, 128),
                (64, 64, 64), (128, 64, 32), (256, 128, 64)]
tab = bancada.rodar_etapa("6. Arquitetura", {str(a): {"camadas": a} for a in ARQUITETURAS})
tab

## Etapa 7 — Regularização

- **L2 (`alpha`)**: penaliza pesos grandes, reduzindo o sobreajuste;
- **paciência** da parada antecipada: quantas épocas sem melhora esperar antes de parar.

In [ ]:
variacoes = {f"L2 = {a:g}": {"l2": a} for a in [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]}
tab = bancada.rodar_etapa("7a. Regularização L2", variacoes)
tab

In [ ]:
tab = bancada.rodar_etapa("7b. Paciência", {f"paciência = {p}": {"paciencia": p} for p in [5, 10, 20, 30]})
tab

## Etapa 8 — Função de ativação

In [ ]:
tab = bancada.rodar_etapa("8. Ativação", {a: {"ativacao": a} for a in ["relu", "tanh", "logistic"]})
tab

## Etapa 9 — Otimização

Otimizador e taxa de aprendizado são testados juntos, porque a melhor taxa depende do otimizador. Depois, o tamanho do lote e a redução da taxa de aprendizado quando a perda de validação estaciona (*ReduceLROnPlateau*).

In [ ]:
variacoes = {f"adam, lr = {lr:g}": {"otimizador": "adam", "lr": lr} for lr in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]}
variacoes.update({f"sgd + nesterov, lr = {lr:g}": {"otimizador": "sgd", "lr": lr} for lr in [3e-3, 1e-2, 3e-2, 1e-1]})
tab = bancada.rodar_etapa("9a. Otimizador e taxa de aprendizado", variacoes)
tab

In [ ]:
variacoes = {f"batch = {b}": {"batch_size": b} for b in [32, 64, 128, 256, 512, 1024]}
variacoes.update({f"batch = {b} + reduzir lr": {"batch_size": b, "reduzir_lr": True} for b in [64, 256, 1024]})
tab = bancada.rodar_etapa("9b. Lote e redução da taxa", variacoes)
tab

## Etapa 10 — Balanceamento das classes

Técnicas para lidar com o desbalanceamento (76% × 24%), aplicadas **somente nos dados de treino** de cada fold:

- **class_weight:** cada exemplo da classe minoritária pesa mais na função de perda;
- **SMOTE:** cria exemplos sintéticos da classe minoritária interpolando vizinhos;
- **oversampling / undersampling:** repete exemplos da minoritária / descarta exemplos da majoritária.

Essas técnicas deslocam as probabilidades para cima: em geral **aumentam o recall e reduzem a precisão** no limiar 0,5, sem melhorar muito a ROC-AUC. A Etapa 12 mostra que o mesmo equilíbrio pode ser obtido apenas ajustando o limiar.

In [ ]:
tab = bancada.rodar_etapa("10. Balanceamento", {
    "nenhum": {"balanceamento": "nenhum"},
    "class_weight (pesos)": {"balanceamento": "class_weight"},
    "SMOTE": {"balanceamento": "smote"},
    "random oversampling": {"balanceamento": "oversampling"},
    "random undersampling": {"balanceamento": "undersampling"},
})
tab

## Etapa 11 — Busca aleatória dos hiperparâmetros da rede

A busca gulosa ajusta um aspecto por vez e pode perder combinações que só funcionam juntas (ex.: rede maior **com** mais regularização e taxa menor). Aqui sorteamos 30 combinações de arquitetura, L2, ativação, taxa de aprendizado e lote, mantendo o pré-processamento escolhido.

In [ ]:
espaco = {
    "camadas": [(64,), (128,), (256,), (64, 32), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "l2": [1e-4, 1e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0],
    "ativacao": ["relu", "tanh"],
    "lr": [1e-4, 3e-4, 1e-3, 3e-3],
    "batch_size": [64, 128, 256, 512],
}
sorteios = list(ParameterSampler(espaco, n_iter=30, random_state=RANDOM_STATE))
variacoes = {"atual": {}}
for i, s in enumerate(sorteios, 1):
    variacoes[f"#{i:02d} {s['camadas']} {s['ativacao']} L2={s['l2']:g} lr={s['lr']:g} b={s['batch_size']}"] = {
        **s, "otimizador": "adam"}
tab = bancada.rodar_etapa("11. Busca aleatória", variacoes)
tab.head(10)

In [ ]:
print(f"Tempo total dos experimentos: {(time.time() - inicio_total) / 60:.1f} min")
print("\nMelhor configuração encontrada:")
melhor_cfg = dict(bancada.melhor_cfg)
pd.Series(melhor_cfg, name="valor").to_frame()

## Evolução ao longo das etapas

A melhor ROC-AUC (média dos 5 folds) ao fim de cada etapa. As barras de erro são o desvio-padrão entre folds. Diferenças menores que ~0,001 estão dentro da variação natural e não devem ser interpretadas como ganhos reais.

In [ ]:
import os
os.makedirs("resultados", exist_ok=True)
resultados = bancada.resultados()
resultados.drop(columns=["cfg"]).to_csv("resultados/resultados_experimentos.csv", index=False)

etapas_adotadas = [e for e in resultados["etapa"].unique() if not e.startswith(("4a", "4b"))]
evolucao = (resultados[resultados["etapa"].isin(etapas_adotadas)]
            .sort_values("roc_auc", ascending=False).groupby("etapa", sort=False).head(1)
            .set_index("etapa").loc[etapas_adotadas])
evolucao[["experimento", "roc_auc", "roc_auc_dp", "pr_auc", "acuracia", "precisao", "recall", "f1", "mcc"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
pos = np.arange(len(evolucao))
ax.errorbar(pos, evolucao["roc_auc"], yerr=evolucao["roc_auc_dp"], color=AZUL, marker="o",
            markersize=7, capsize=4, label="ROC-AUC da melhor variação da etapa")
ax.set_xticks(pos, [e.split(". ")[1] if ". " in e else e for e in evolucao.index], rotation=35, ha="right")
ax.set_ylabel("ROC-AUC (validação cruzada)")
ax.set_title("Evolução da ROC-AUC ao longo das etapas da bancada")
for i, v in enumerate(evolucao["roc_auc"]):
    if i in (0, len(evolucao) - 1):
        ax.annotate(f"{v:.4f}", (i, v), textcoords="offset points", xytext=(0, 10), ha="center", color=CINZA_TEXTO)
plt.tight_layout()
plt.show()

## Etapa 12 — Limiar de decisão

A rede devolve uma probabilidade; o limiar 0,5 é só uma convenção. Usando as **previsões fora-do-fold** da melhor configuração (cada exemplo do treino foi previsto por um modelo que não o viu), escolhemos o limiar que maximiza:

- o **F1** (equilíbrio entre precisão e recall da classe `>50K`);
- a **acurácia**.

Isso é feito só com dados de treino, então o teste continua intocado.

In [ ]:
oof = bancada.oof_melhor()
limiares = np.round(np.arange(0.05, 0.951, 0.01), 2)
curvas = pd.DataFrame([bc.calcular_metricas(y_treino, oof, t) for t in limiares], index=limiares)

limiar_f1 = curvas["f1"].idxmax()
limiar_acc = curvas["acuracia"].idxmax()
print(f"Limiar que maximiza o F1:       {limiar_f1:.2f}")
print(f"Limiar que maximiza a acurácia: {limiar_acc:.2f}")
curvas.loc[sorted({0.5, limiar_f1, limiar_acc}), ["acuracia", "precisao", "recall", "f1", "mcc", "acuracia_bal"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
for col, cor, nome in [("precisao", AZUL, "precisão"), ("recall", LARANJA, "recall"),
                       ("f1", VERDE_AGUA, "F1"), ("acuracia", AMARELO, "acurácia")]:
    ax.plot(curvas.index, curvas[col], color=cor, label=nome)
ax.axvline(0.5, color=CINZA_TEXTO, lw=1, ls=":")
ax.axvline(limiar_f1, color=VERDE_AGUA, lw=1, ls="--")
ax.annotate(f"máx. F1 ({limiar_f1:.2f})", (limiar_f1, 0.32), xytext=(4, 0), textcoords="offset points", color=CINZA_TEXTO)
ax.annotate("0,5", (0.5, 0.25), xytext=(4, 0), textcoords="offset points", color=CINZA_TEXTO)
ax.set(xlabel="limiar de decisão", ylabel="métrica (validação cruzada)", ylim=(0.2, 1.0),
       title="Métricas em função do limiar (previsões fora-do-fold)")
ax.legend(loc="lower left", ncol=4, frameon=False)
plt.tight_layout()
plt.show()

## Estabilidade da melhor configuração

Repetimos a validação cruzada da melhor configuração com outras sementes (novas divisões dos folds e novas inicializações da rede). A variação entre sementes mostra quanto das diferenças entre experimentos é ruído.

In [ ]:
estab = []
for semente in [1, 2, 3, 4, 5]:
    r = bc.avaliar_cv(X_treino, y_treino, melhor_cfg, n_folds=5, seed=semente)
    estab.append({"semente": semente, **{m: r[m] for m in ["roc_auc", "pr_auc", "acuracia", "f1"]}})
estab = pd.DataFrame(estab).set_index("semente")
print(estab.round(4))
print("\nMédia ± desvio-padrão:")
print((estab.mean().round(4).astype(str) + " ± " + estab.std().round(4).astype(str)).to_string())

## Modelo final e avaliação no teste

O modelo final usa a melhor configuração, treinada em todo o conjunto de treino. Como a inicialização aleatória dos pesos muda um pouco o resultado, treinamos **5 redes com sementes diferentes** e usamos a **média das probabilidades** (ensemble de sementes) — uma técnica simples que costuma melhorar e estabilizar as métricas.

In [ ]:
SEMENTES = [0, 1, 2, 3, 4]
modelos, probas_teste = [], []
for s in SEMENTES:
    prep, rede, n_ep = bc.treinar_modelo(X_treino, y_treino, melhor_cfg, seed=s)
    modelos.append((prep, rede))
    probas_teste.append(bc.prever_proba(prep, rede, X_teste))
    print(f"semente {s}: {n_ep} épocas, ROC-AUC no teste = {bc.calcular_metricas(y_teste, probas_teste[-1])['roc_auc']:.4f}")
proba_ensemble = np.mean(probas_teste, axis=0)

In [ ]:
# Curva de aprendizado de uma das redes: perda no conjunto de parada antecipada a cada época
rede = modelos[0][1]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.arange(1, len(rede.curva_perda_) + 1), rede.curva_perda_, color=AZUL, label="perda de validação")
melhor_ep = int(np.argmin(rede.curva_perda_)) + 1
ax.axvline(melhor_ep, color=CINZA_TEXTO, ls="--", lw=1)
ax.annotate(f"melhor época ({melhor_ep}): pesos restaurados", (melhor_ep, max(rede.curva_perda_)),
            xytext=(6, -8), textcoords="offset points", color=CINZA_TEXTO, va="top")
ax.set(xlabel="época", ylabel="log loss", title="Parada antecipada da rede final (semente 0)")
plt.tight_layout()
plt.show()

In [ ]:
linhas = {
    "MLP final, 1 rede (média das 5 sementes), limiar 0,5":
        pd.DataFrame([bc.calcular_metricas(y_teste, p) for p in probas_teste]).mean(),
    "MLP final, ensemble de 5 sementes, limiar 0,5": pd.Series(bc.calcular_metricas(y_teste, proba_ensemble)),
    f"MLP final, ensemble, limiar máx. F1 ({limiar_f1:.2f})": pd.Series(bc.calcular_metricas(y_teste, proba_ensemble, limiar_f1)),
    f"MLP final, ensemble, limiar máx. acurácia ({limiar_acc:.2f})": pd.Series(bc.calcular_metricas(y_teste, proba_ensemble, limiar_acc)),
}
tabela_teste = pd.DataFrame(linhas).T[bc.METRICAS]
tabela_teste.round(4)

In [ ]:
pred_final = (proba_ensemble >= limiar_f1).astype(int)
print(classification_report(y_teste, pred_final, target_names=["<=50K", ">50K"], digits=4))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(10, 4))
for ax, limiar, titulo in [(eixos[0], 0.5, "limiar 0,5"), (eixos[1], limiar_f1, f"limiar máx. F1 ({limiar_f1:.2f})")]:
    ConfusionMatrixDisplay.from_predictions(
        y_teste, (proba_ensemble >= limiar).astype(int), display_labels=["<=50K", ">50K"],
        cmap="Blues", colorbar=False, ax=ax)
    ax.grid(False)
    ax.set_title(f"Matriz de confusão no teste — {titulo}")
plt.tight_layout()
plt.show()

## Comparação com outros modelos

Para situar o resultado da rede, avaliamos no **mesmo teste**:

- a **MLP da linha de base** (configuração inicial, sem as melhorias da bancada);
- a **regressão logística** do notebook de PCA (`LabelEncoder` + `StandardScaler`) e a mesma com *one-hot*;
- a **Random Forest** do tutorial (`LabelEncoder`);
- o **HistGradientBoosting**, um modelo de *boosting* de árvores, que costuma ser a referência mais forte em dados tabulares.

In [ ]:
def pipeline_sklearn(modelo, codificacao="onehot", escala=True):
    if codificacao == "onehot":
        cod = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        cod = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("cod", cod)]
                   + ([("esc", StandardScaler())] if escala and codificacao != "onehot" else []))
    num = StandardScaler() if escala else "passthrough"
    colunas = ColumnTransformer([("num", num, make_column_selector(dtype_include="number")),
                                 ("cat", cat, make_column_selector(dtype_exclude="number"))])
    return Pipeline([("colunas", colunas), ("modelo", modelo)])

referencias = {
    "Regressão logística (label, como no notebook de PCA)":
        pipeline_sklearn(LogisticRegression(max_iter=2000), "label"),
    "Regressão logística (one-hot)":
        pipeline_sklearn(LogisticRegression(max_iter=2000), "onehot"),
    "Random Forest (label, como no tutorial)":
        pipeline_sklearn(RandomForestClassifier(random_state=2022, n_jobs=-1), "label", escala=False),
    "HistGradientBoosting (label)":
        pipeline_sklearn(HistGradientBoostingClassifier(random_state=RANDOM_STATE), "label", escala=False),
}
probas_ref = {}
for nome, modelo in referencias.items():
    modelo.fit(X_treino, y_treino)
    probas_ref[nome] = modelo.predict_proba(X_teste)[:, 1]

# MLP da linha de base, treinada no treino completo
prep0, rede0, _ = bc.treinar_modelo(X_treino, y_treino, bc.CONFIG_BASE, seed=0)
probas_ref["MLP linha de base (label + standard, 64-32)"] = bc.prever_proba(prep0, rede0, X_teste)
probas_ref["MLP final (ensemble de 5 sementes)"] = proba_ensemble

comparacao = pd.DataFrame({nome: bc.calcular_metricas(y_teste, p) for nome, p in probas_ref.items()}).T
comparacao = comparacao[bc.METRICAS].sort_values("roc_auc", ascending=False)
print("Todas as métricas com limiar 0,5 (exceto ROC-AUC, PR-AUC e log loss, que não dependem do limiar)")
comparacao.round(4)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.6))
destaque = {"MLP final (ensemble de 5 sementes)": AZUL,
            "MLP linha de base (label + standard, 64-32)": LARANJA,
            "HistGradientBoosting (label)": VERDE_AGUA}
for nome, cor in destaque.items():
    p = probas_ref[nome]
    fpr, tpr, _ = roc_curve(y_teste, p)
    eixos[0].plot(fpr, tpr, color=cor, label=f"{nome} (AUC {comparacao.loc[nome, 'roc_auc']:.4f})")
    prec, rec, _ = precision_recall_curve(y_teste, p)
    eixos[1].plot(rec, prec, color=cor, label=f"{nome} (AP {comparacao.loc[nome, 'pr_auc']:.4f})")
eixos[0].plot([0, 1], [0, 1], color=CINZA_TEXTO, lw=1, ls=":")
eixos[0].set(xlabel="taxa de falsos positivos", ylabel="taxa de verdadeiros positivos (recall)", title="Curva ROC — teste")
eixos[1].axhline(y_teste.mean(), color=CINZA_TEXTO, lw=1, ls=":")
eixos[1].set(xlabel="recall", ylabel="precisão", title="Curva precisão × recall — teste")
for ax in eixos:
    ax.legend(loc="lower left" if ax is eixos[1] else "lower right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()